# Sistema de Leitura de PDFs e Perguntas e Respostas com IA Generativa


#Objetivo

Desenvolver um sistema de leitura, processamento e consulta a documentos PDF utilizando técnicas de Processamento de Linguagem Natural (PLN), embeddings semânticos, bancos de dados vetoriais e Modelos de Linguagem de Grande Escala (LLMs).

Como documento de referência, foi utilizado um trabalho de iniciação científica de minha autoria, intitulado “Mapeamento e Estimação da Subnotificação de Casos e Mortes por Tuberculose nas Microrregiões do Brasil, 2012 a 2020”.

A proposta consiste em explorar como técnicas de IA Generativa podem facilitar a consulta a documentos científicos, permitindo recuperar informações relevantes por meio de perguntas em linguagem natural e gerar respostas contextualizadas com base no conteúdo do documento.

#Metodologia

O projeto foi desenvolvido em etapas:

**Extração de texto:** leitura do arquivo PDF e obtenção de seu conteúdo textual.

**Fragmentação do documento (chunking):** divisão do texto em trechos menores, com parâmetros de tamanho e sobreposição.

**Geração de embeddings:** transformação dos fragmentos em vetores numéricos por meio do modelo all-MiniLM-L6-v2 disponível na plataforma Hugging Face.

**Indexação vetorial:** armazenamento dos vetores e documentos no ChromaDB para permitir consultas semânticas.

**Recuperação de informações:** utilização de busca semântica e da técnica Maximal Marginal Relevance (MMR) para selecionar trechos relevantes para as perguntas.

**Construção de prompts:** organização das instruções enviadas ao modelo de linguagem por meio de templates.

**Geração de respostas com LLM:** utilização do modelo Qwen/Qwen2.5-1.5B-Instruct, disponibilizado pela Hugging Face, para gerar respostas com base no contexto recuperado do documento.

**Geração de respostas:** utilização do contexto recuperado para responder às perguntas sobre o documento, seguindo a abordagem Retrieval-Augmented Generation (RAG).

In [1]:
# Instala o pacote watermark.
# Esse pacote é usado para gravar as versões dos pacotes usados neste  notebook.
!pip install -q -U watermark

In [2]:
%env TOKENIZERS_PARALLELISM=True

env: TOKENIZERS_PARALLELISM=True


In [3]:
%env TF_CPP_MIN_LOG_LEVEL=3

env: TF_CPP_MIN_LOG_LEVEL=3


In [5]:
# Imports
import pypdf
import chromadb
import urllib3
import accelerate
import sentence_transformers
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

#Extraindo Dados de Texto de Arquivos PDF

In [6]:
#Cria um objeto para carregar o arquivo PDF
doc_loader = PyPDFLoader("/content/Relatorio_Final ProjetoIC_102872021.pdf")

In [7]:
type(doc_loader)

langchain_community.document_loaders.pdf.PyPDFLoader

In [8]:
#Carrega o arquivo PDF
paginas = doc_loader.load()

In [9]:
paginas

[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-11-10T15:09:26-03:00', 'author': 'Guilherme Lopes de Oliveira', 'moddate': '2023-11-10T15:09:26-03:00', 'source': '/content/Relatorio_Final ProjetoIC_102872021.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1'}, page_content='CENTRO FEDERAL DE EDUCAÇÃO TECNOLÓGICA DE MINAS GERAIS \nDIRETORIA DE PESQUISA E PÓS-GRADUAÇÃO \nPROGRAMA INSTITUCIONAL DE INICIAÇÃO CIENTÍFICA \n \n \n \n \n \n \n \n \nIGOR ANDRÉ SILVA COELHO \n \n \n \n \n \n \n \n \n \nMAPEAMENTO E ESTIMAÇÃO DA SUBNOTIFICAÇÃO DE CASOS \nE MORTES POR TUBERCULOSE NAS MICRORREGIÕES DO \nBRASIL, 2012 A 2020 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nBELO HORIZONTE - MG \n2023'),
 Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-11-10T15:09:26-03:00', 'author': 'Guilherme Lopes de Oliveira', 'moddate': '2023-11-10T15:09:26-03:00', 'source': '/conte

In [10]:
# Visualizando o conteúdeo de apenas uma página
unico = paginas[2]

In [11]:
print("Conteúdo da Página:", unico.page_content[0:700])

Conteúdo da Página: RESUMO 
 
A tuberculose (TB) ainda é um sério problema de saúde pública no mundo. A não 
detecção dos casos e mortes por TB é um import ante desafio a ser enfrentado, pois 
contribui para a manutenção da cadeia de transmissão, impede o tratamento eficaz e 
colabora para a subestimação da magnitude do problema. Este projeto tem o objetivo 
de estimar e corrigir a subnotificação nas taxas de in cidência e de mortalidade 
associadas à TB nas microrregiões do Brasil nos anos de 2003 a 2020. Para tal, será 
feito uso de modelos Bayesianos que permitem a modelagem conjunta do processo 
de ocorrência e de notificação dos casos. Na literatura, existem poucos estudos sobre 
a tuberculose de abrangênc


In [12]:
print("Metadados da Página:", unico.metadata)

Metadados da Página: {'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-11-10T15:09:26-03:00', 'author': 'Guilherme Lopes de Oliveira', 'moddate': '2023-11-10T15:09:26-03:00', 'source': '/content/Relatorio_Final ProjetoIC_102872021.pdf', 'total_pages': 30, 'page': 2, 'page_label': '3'}


## Divisão dos Dados de Texto em Chunks

In [13]:
# Cria o separador de chunks de texto
separador = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 20)

Parâmetros de divisão do texto

**chunk_size = 1000:** Estabelece o tamanho máximo de cada fragmento de texto em 1.000 caracteres. Essa configuração permite dividir documentos extensos em partes menores, facilitando o processamento das informações.

**chunk_overlap = 20:** Define uma sobreposição de 20 caracteres entre fragmentos consecutivos. Dessa forma, o final de um trecho é parcialmente repetido no início do próximo, ajudando a preservar a continuidade das informações durante a divisão.

Por que dividir o texto em chunks?

A fragmentação de documentos é uma etapa importante no processamento de textos extensos, especialmente em aplicações que utilizam modelos de linguagem (LLMs). Entre suas principais finalidades, destacam-se:

Compatibilidade com modelos de linguagem: dividir o conteúdo em partes menores ajuda a respeitar os limites de entrada dos modelos, considerando a quantidade de tokens que podem processar por vez.

Indexação e recuperação de informações: fragmentos menores facilitam a organização, a indexação e a localização de trechos relevantes em mecanismos de busca e sistemas baseados em RAG (Retrieval-Augmented Generation).

Preservação do contexto: a sobreposição entre fragmentos reduz o risco de separar informações que dependem umas das outras, contribuindo para que o sistema recupere trechos mais completos e coerentes.

Esses parâmetros podem ser ajustados conforme as características dos documentos e os objetivos da aplicação. O tamanho dos fragmentos e o nível de sobreposição influenciam tanto a qualidade da recuperação de informações quanto o custo computacional do processamento.

In [14]:
#Aplica o  objeto e extrai os chunks(Documentos)
docs = separador.split_documents(paginas)

In [15]:
print("Total de Chunks (Documentos):", len(docs))

Total de Chunks (Documentos): 49


In [16]:
print("Conteúdo do Último Chunck (Documento):", docs[48])

Conteúdo do Último Chunck (Documento): page_content='Correcting Under -Reporting in Count Data. Journal of the American Statistical 
Association, 114(528), 1481–1492.  
 
[6] Oliveira, G.L., Loschi, R.H., et al.  (2021). Bias correction in clustered 
underreported data. Bayesian Analysis, Advance Publication 1-32. 
 
[7] Silva, G.D.M., Bartholomay, L., Cruz, O.G. and Garcia, L.P. (2017). Evaluation of 
data quality, timeliness and acceptability of the tuberculosis surveillance system in 
Brazil’s micro-regions. Ciência & Saúde Coletiva, 22(10), 3307–3319.' metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-11-10T15:09:26-03:00', 'author': 'Guilherme Lopes de Oliveira', 'moddate': '2023-11-10T15:09:26-03:00', 'source': '/content/Relatorio_Final ProjetoIC_102872021.pdf', 'total_pages': 30, 'page': 29, 'page_label': '30'}


## Carregando os Vetores dos Dados de Texto no Banco de Dados Vetorial

O código utiliza um banco de dados vetorial (Vector Database) para realizar buscas semânticas, recuperando os trechos de documentos mais relevantes para uma pergunta com base na similaridade de significado entre o conteúdo armazenado e a consulta realizada.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

https://www.trychroma.com/

In [17]:
# Cria o banco de dados vetorial
vectordb = Chroma.from_documents(documents = docs,
                                 embedding = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2"),
                                 persist_directory = "dsavectordb/chroma/")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Chroma.from_documents(documents=docs):** Cria um banco de dados vetorial a partir dos documentos contidos na variável docs, permitindo organizar e armazenar os fragmentos de texto para consultas e recuperação de informações relevantes.

**HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"):** Utiliza o modelo all-MiniLM-L6-v2, disponibilizado pela Hugging Face, para converter os textos em vetores numéricos que representam suas características semânticas. Essas representações permitem comparar a similaridade de significado entre diferentes textos e consultas.

**persist_directory="dsavectordb/chroma/":** Define o caminho do diretório em que os dados do banco vetorial serão armazenados de forma persistente, possibilitando reutilizar os documentos indexados em execuções futuras, sem precisar reconstruir o banco a cada vez.

In [18]:
#Total de coleções no vector db
vectordb._collection.count()

49

#Testando os parâmetros da Busca Vetorial

In [19]:
#Define uma pergunta
pergunta = "Qual a proporção de casos de TB notificados no país?"

In [20]:
#Realiza a busca vetorial
pontos_relevantes = vectordb.max_marginal_relevance_search(pergunta, k = 2, fetch_k = 10)

**max_marginal_relevance_search():** Executa uma busca no banco de dados vetorial utilizando a técnica de Relevância Marginal Máxima (MMR — Maximal Marginal Relevance). O método seleciona documentos que sejam relevantes para a pergunta, buscando também reduzir a repetição de informações entre os resultados. Dessa forma, a consulta pode retornar trechos complementares, em vez de apresentar apenas documentos com conteúdos muito semelhantes.

**Parâmetros:**

**pergunta:** Texto da pergunta utilizada como consulta para localizar documentos semanticamente relacionados ao conteúdo solicitado.

**k=2:** Determina a quantidade final de documentos que serão retornados pela busca, neste caso, dois fragmentos de texto.

**fetch_k=10:** Define a quantidade inicial de documentos candidatos considerados pelo algoritmo antes da seleção final.

In [21]:
print(pontos_relevantes[0])

page_content='No Brasil, a vigilância epidemiológica, particularmente o Sistema de 
Informação de Agravos de Notifica ção (SINAN), tem se mostrado uma importante 
ferramenta para auxiliar em ações de controle de doenças infecciosas. Mesmo assim, 
estima-se que a proporção de casos de TB notificados no país era de 87% em 2017 
[1]. A alta rotatividade dos profissionais de saú de e a demora para atendimento nos 
serviços de saúde são alguns dos desafios para a melhoria da notificação de casos 
no país. Também, a doença permanece pouco conhecida por parte da população, 
incluindo-se profissionais de saúde que atuam na atenção primár ia. Sabe-se que a 
desinformação sobre a TB também está associada à subnotificação de casos.  
O sub-registro (subnotificação) é um fenômeno comum, sobretudo, em países 
menos desenvolvidos e leva à produção de estimativas viesadas para as taxas de 
incidência de agravos como a tuberculose. O problema se torna ainda mais evidente' metadata={'creationdate': '20

#Definindo o LLM

https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

In [22]:
#Define o nome do LLM conforme consta no Hugging Face
nome_modelo_llm = "Qwen/Qwen2.5-1.5B-Instruct"

In [23]:
#Carrega o modelo
modelo = AutoModelForCausalLM.from_pretrained(nome_modelo_llm,
                                              torch_dtype = "auto",
                                              device_map = "auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [24]:
#Carrega o tokenizador do modelo
tokenizer = AutoTokenizer.from_pretrained(nome_modelo_llm)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

#Configurando o Contexto

In [25]:
#Define uma pergunta
pergunta = "Qual a proporção de casos de TB notificados no país?"

In [26]:
#Extrai o contexto da pergunta
contexto = vectordb.max_marginal_relevance_search(pergunta, k = 2, fetch_k = 5)

#Configura o Prompt

In [27]:
#Cria o prompt
prompt = f"""
Você é um assistente especialista. Você usa o contexto fornecido como sua base de conhecimento complementar para responder à pergunta.
context = {contexto}
question = {pergunta}
answer =
"""

In [28]:
# Cria a lista de mensagens de systema e de usuário
messages = [
    {"role": "system", "content": "Você é um assistente especializado em análise de documentos e recuperação de informações."},
    {"role": "user", "content": prompt}
]

#Tokenização do Prompt

In [29]:
#Aplica o chat template
texto = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)

Os Prompt Templates  são estruturas utilizadas para organizar e personalizar as instruções enviadas aos modelos de linguagem (LLMs), permitindo adaptar as entradas de acordo com a tarefa desejada.

Um template de prompt contém um texto predefinido com variáveis que podem ser substituídas por valores específicos durante a execução. Isso possibilita gerar prompts dinamicamente, mantendo um padrão e evitando a necessidade de escrever manualmente cada solicitação.

Esse recurso facilita a construção e a manutenção de aplicações com LLMs, permitindo reutilizar instruções, adaptar o conteúdo às diferentes consultas e testar estratégias para aprimorar a qualidade e a consistência das respostas.

In [30]:
texto

"<|im_start|>system\nVocê é um assistente especializado em análise de documentos e recuperação de informações.<|im_end|>\n<|im_start|>user\n\nVocê é um assistente especialista. Você usa o contexto fornecido como sua base de conhecimento complementar para responder à pergunta.\ncontext = [Document(metadata={'page_label': '4', 'moddate': '2023-11-10T15:09:26-03:00', 'page': 3, 'total_pages': 30, 'creator': 'Microsoft® Word 2013', 'producer': 'Microsoft® Word 2013', 'creationdate': '2023-11-10T15:09:26-03:00', 'source': '/content/Relatorio_Final ProjetoIC_102872021.pdf', 'author': 'Guilherme Lopes de Oliveira'}, page_content='No Brasil, a vigilância epidemiológica, particularmente o Sistema de \\nInformação de Agravos de Notifica ção (SINAN), tem se mostrado uma importante \\nferramenta para auxiliar em ações de controle de doenças infecciosas. Mesmo assim, \\nestima-se que a proporção de casos de TB notificados no país era de 87% em 2017 \\n[1]. A alta rotatividade dos profissionais de s

In [31]:
#Aplica a tokenização
model_inputs = tokenizer([texto], return_tensors = "pt").to(modelo.device)

In [32]:
model_inputs

{'input_ids': tensor([[151644,   8948,    198,  69286,   3958,   4443,   7789,   6817,  32297,
          25846,    976, 138482,    409,  99027,    384,   1395,    454,  74744,
            409,  64066,     13, 151645,    198, 151644,    872,    271,  69286,
           3958,   4443,   7789,   6817,  32297,   9087,     13,  80797,  33715,
            297,  76743,  56009,    757,   5249,   7953,  19345,   2331,    409,
          70483,  15027,  22766,    277,   3348,  64034,   3784,    817,  59910,
            624,   2147,    284,    508,   7524,  54436,  12854,   2893,   6106,
           1210,    364,     19,    516,    364,   2593,   1028,   1210,    364,
             17,     15,     17,     18,     12,     16,     16,     12,     16,
             15,     51,     16,     20,     25,     15,     24,     25,     17,
             21,     12,     15,     18,     25,     15,     15,    516,    364,
           2893,   1210,    220,     18,     11,    364,   5035,  21655,   1210,
            22

#Gerando Resposta com o LLM

In [33]:
#Gera resposta com o LLM
generated_ids = modelo.generate(**model_inputs, max_new_tokens = 512)

In [34]:
generated_ids

tensor([[151644,   8948,    198,  69286,   3958,   4443,   7789,   6817,  32297,
          25846,    976, 138482,    409,  99027,    384,   1395,    454,  74744,
            409,  64066,     13, 151645,    198, 151644,    872,    271,  69286,
           3958,   4443,   7789,   6817,  32297,   9087,     13,  80797,  33715,
            297,  76743,  56009,    757,   5249,   7953,  19345,   2331,    409,
          70483,  15027,  22766,    277,   3348,  64034,   3784,    817,  59910,
            624,   2147,    284,    508,   7524,  54436,  12854,   2893,   6106,
           1210,    364,     19,    516,    364,   2593,   1028,   1210,    364,
             17,     15,     17,     18,     12,     16,     16,     12,     16,
             15,     51,     16,     20,     25,     15,     24,     25,     17,
             21,     12,     15,     18,     25,     15,     15,    516,    364,
           2893,   1210,    220,     18,     11,    364,   5035,  21655,   1210,
            220,     18,    

In [35]:
#Descompacta as respostas
#Objetivo: Extrair apenas os tokens gerados pelo modelo (ou seja, a parte da saída que vem após os tokens de entrada)

generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids,generated_ids)]

In [36]:
generated_ids

[tensor([  1912,  76926,    469,    297,  53426,  56009,    757,   5249,     11,
            264,  70908,   5903,    409,  57533,    409,  30080,    537,  90762,
            902,  41925,    976,    220,     17,     15,     16,     22,  21679,
            409,  94424,  76055,    220,     23,     22,  14360, 151645],
        device='cuda:0')]

In [37]:
#Aplica o decode para obter o texto gerado
response = tokenizer.batch_decode(generated_ids, skip_special_tokens = True)[0]

In [38]:
print(response)

De acordo com o documento fornecido, a proporção de casos de TB notificados no Brasil em 2017 foi de aproximadamente 87%.


#Sistema de Perguntas e Respostas Usando a Base de Conhecimento

In [40]:
#Define a pergunta
pergunta = "Qual o objetivo da pesquisa?"

#Extrai o contexto da questão (ou seja, executa a busca vetorial)
contexto = vectordb.max_marginal_relevance_search(pergunta, k = 2, fetch_k = 3)

# Cria o prompt
prompt = f"""
Você é um assistente especialista. Você usa o contexto fornecido como sua base de conhecimento compplementar para responder à pergunta.
context = {contexto}
question = {pergunta}
answer =
"""

# Cria a lista de mensagens de systema e de usuário
messages = [
    {"role": "system", "content": "Você é Qwen, criado pela Alibaba Cloud. Você é um assistente especialista."},
    {"role": "user", "content": prompt}
]

#Aplica o chat template
texto = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)

#Aplica a tokenização
model_inputs = tokenizer([texto], return_tensors = "pt").to(modelo.device)

#Gera resposta com o LLM
generated_ids = modelo.generate(**model_inputs, max_new_tokens = 1000)

#Descompacta as respostas
generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids,generated_ids)]

#Aplica o decode para obter o texto gerado
response = tokenizer.batch_decode(generated_ids, skip_special_tokens = True)[0]

print(response)

A pesquisa tem como objetivo analisar o agrupamento de dados de contatos em diferentes períodos de tempo (triênios) e identificar quaisquer problemas ou desafios encontrados durante esse processo. Esta análise visa compreender melhor os padrões e tendências nos registros de contatos e ajudar a tomar decisões informadas para otimizar processos futuros relacionados ao gerenciamento de dados de contato.


In [43]:
#Define a pergunta
pergunta = "Existem dificuldades para registrar e notificar os casos?"

#Extrai o contexto da questão (ou seja, executa a busca vetorial)
contexto = vectordb.max_marginal_relevance_search(pergunta, k = 2, fetch_k = 3)

# Cria o prompt
prompt = f"""
Você é um assistente especialista. Você usa o contexto fornecido como sua base de conhecimento compplementar para responder à pergunta.
context = {contexto}
question = {pergunta}
answer =
"""

# Cria a lista de mensagens de systema e de usuário
messages = [
    {"role": "system", "content": "Você é Qwen, criado pela Alibaba Cloud. Você é um assistente especialista."},
    {"role": "user", "content": prompt}
]

#Aplica o chat template
texto = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)

#Aplica a tokenização
model_inputs = tokenizer([texto], return_tensors = "pt").to(modelo.device)

#Gera resposta com o LLM
generated_ids = modelo.generate(**model_inputs, max_new_tokens = 1000)

#Descompacta as respostas
generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids,generated_ids)]

#Aplica o decode para obter o texto gerado
response = tokenizer.batch_decode(generated_ids, skip_special_tokens = True)[0]

print(response)

Sim, existem algumas dificuldades para registrar e notificar os casos de tuberculose. Segundo o documento, os cenários propostos incluem cenários de consistência dos dados, como o percentual de casos com devido registro. Além disso, há indicações de problemas nos indicadores de oportunidade, como a proporção de casos novos de tuberculose pulmonar que realizaram tratamento diretamente observado.

Aqui estão alguns pontos importantes:

1. Percentual de casos com devido registro:
   - Examinou-se apenas os casos preenchidos, sem preenchimento ou preenchidos com "ignorado" ou outros termos que indicassem a ausência de informação.

2. Indicadores de oportunidade:
   - Propôs indicadores como Oportunidade de notificação, Oportunidade de digitação, Oportunidade de tratamento e Oportunidade de encerramento do caso.

Estes são aspectos críticos na gestão efetiva de dados de saúde pública, especialmente no contexto de doenças endêmicas como a tuberculose. A capacidade de registrar e notificar ad

In [44]:
# Versões dos pacotes usados neste notebook
%reload_ext watermark


In [45]:
%watermark -v -m

Python implementation: CPython
Python version       : 3.13.16
IPython version      : 7.34.0

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 6.6.159+
Machine     : x86_64
Processor   : x86_64
CPU cores   : 2
Architecture: 64bit



In [46]:
%watermark --iversions

accelerate              : 1.9.0
chromadb                : 1.0.15
langchain               : 0.3.27
langchain_community     : 0.3.27
langchain_core          : 0.3.72
langchain_text_splitters: 0.3.9
pypdf                   : 5.9.0
sentence_transformers   : 5.7.0
transformers            : 4.54.1
urllib3                 : 2.8.0



#Conclusão

O desenvolvimento deste projeto permitiu explorar, na prática, a aplicação de técnicas de IA Generativa para processamento e recuperação de informações em documentos científicos. A integração entre extração de texto de PDFs, fragmentação de documentos (chunking), embeddings semânticos, armazenamento vetorial e modelos de linguagem possibilitou a construção de um sistema de perguntas e respostas baseado na abordagem Retrieval-Augmented Generation (RAG).

A utilização do ChromaDB para recuperação de trechos relevantes e do modelo Qwen/Qwen2.5-1.5B-Instruct para geração de respostas demonstrou como diferentes componentes podem ser combinados para facilitar o acesso ao conteúdo de documentos extensos por meio de consultas em linguagem natural.

Além do aprendizado técnico relacionado às ferramentas e aos modelos utilizados, o projeto proporcionou a oportunidade de aplicar conceitos de IA Generativa a um trabalho de iniciação científica de minha autoria, aproximando minha experiência acadêmica em Estatística das aplicações práticas de modelos de linguagem.

Como possibilidades de aprimoramento, destacam-se a avaliação sistemática da qualidade das respostas, a análise da relevância dos documentos recuperados, a inclusão de referências aos trechos utilizados na geração das respostas e a implementação de mecanismos para identificar perguntas que não possam ser respondidas com base no documento.

Dessa forma, o projeto constitui uma aplicação prática dos conceitos de processamento de documentos, recuperação semântica e geração de texto, estabelecendo uma base para o desenvolvimento de sistemas mais robustos de consulta inteligente a documentos científicos e corporativos.